# 06b - Corrected F1 Metric

Recomputes WeakOverlap F1 using a fairer metric:

**Old:** LLM tags vs ALL student weak skills
**New:** LLM tags vs student weak skills THAT THIS PROBLEM ACTUALLY TESTS

testable_weak_skills = student_weak_skills ∩ problem_required_skills

In [ ]:
import ast
import pandas as pd
import numpy as np
from pathlib import Path

from lib.experiment_utils import load_best_attempts_df
from lib.mental_model import (
    load_skill_map,
    calculate_student_profile,
    get_weak_skills,
)

ROOT = Path('.').resolve()
# Ensure ROOT is the project root if running from experiments/
if not (ROOT / 'lib').exists() and (ROOT.parent / 'lib').exists():
    ROOT = ROOT.parent

WEAK_THRESHOLD = 0.6

# Load best attempts
best_attempts_df = load_best_attempts_df()

# Load skill map (problem -> required skills)
skill_map, all_skills_from_file = load_skill_map()
# Ensure all_skills is properly handled if load_skill_map returns a tuple
all_skills = sorted(all_skills_from_file)

# Load batch results
batch_results_path = ROOT / 'results' / '06_batch_30students' / 'batch_comparison_30students.csv'
batch_df = pd.read_csv(batch_results_path)

print(f"Batch results: {len(batch_df)} rows, {batch_df['SubjectID'].nunique()} students")
print(f"Skill map: {len(skill_map)} problems, {len(all_skills)} unique skills")

In [ ]:
# Build weak skills for every student in the batch
student_weak = {}
for sid in batch_df['SubjectID'].unique():
    # calculate_student_profile returns a dict of skill -> mastery
    profile = calculate_student_profile(sid, best_attempts_df, skill_map, all_skills)
    # get_weak_skills returns list of (skill, mastery) tuples
    weak_pairs = get_weak_skills(profile, threshold=WEAK_THRESHOLD)
    student_weak[sid] = set(s for s, m in weak_pairs)

# Build problem required skills
problem_required = {}
for pid, skills in skill_map.items():
    problem_required[str(pid)] = set(skills)

# Sanity check against NumWeakSkills column in CSV
print("Sanity checks:")
for sid in batch_df['SubjectID'].unique():
    expected = batch_df[batch_df['SubjectID'] == sid]['NumWeakSkills'].iloc[0]
    actual = len(student_weak.get(sid, set()))
    if expected != actual:
        print(f"  WARNING: Student {sid} expected {expected} weak skills, got {actual}")

print(f"\nBuilt weak skills for {len(student_weak)} students")
print(f"Built required skills for {len(problem_required)} problems")

# Show a few examples
for sid in list(student_weak.keys())[:3]:
    print(f"  Student {sid}: {student_weak[sid]}")

In [ ]:
def compute_f1(predicted_tags, ground_truth):
    """Compute precision, recall, F1 between two sets."""
    if not ground_truth:
        # If no ground truth (no testable weak skills), we should not 
        # return a high score if predicted is empty (no signal), or 
        # if only a few predicted. Instead, return zero.
        return 0.0, 0.0, 0.0 
    
    if not predicted_tags:
        return 0.0, 0.0, 0.0

    overlap = predicted_tags & ground_truth
    precision = len(overlap) / len(predicted_tags)
    recall = len(overlap) / len(ground_truth)
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1


def parse_kc_tags(tag_str):
    """Parse KC tags from CSV string like \"['If/Else', 'For']\" """
    try:
        # Check if actually a string list expression
        if pd.isna(tag_str) or not isinstance(tag_str, str):
            return set()
        tags = ast.literal_eval(tag_str)
        return set(tags) if isinstance(tags, list) else set()
    except:
        return set()

In [ ]:
results = []

for _, row in batch_df.iterrows():
    sid = row['SubjectID']
    pid = str(row['ProblemID'])

    weak = student_weak.get(sid, set())
    required = problem_required.get(pid, set())
    testable_weak = weak & required  # THE FIX

    baseline_tags = parse_kc_tags(row['Baseline_KCTags'])
    enriched_tags = parse_kc_tags(row['Enriched_KCTags'])

    # Old metric (for comparison)
    _, _, old_bl_f1 = compute_f1(baseline_tags, weak)
    _, _, old_en_f1 = compute_f1(enriched_tags, weak)

    # New corrected metric (testable_weak_skills = student_weak_skills ∩ problem_required_skills)
    bl_p, bl_r, new_bl_f1 = compute_f1(baseline_tags, testable_weak)
    en_p, en_r, new_en_f1 = compute_f1(enriched_tags, testable_weak)

    results.append({
        'SubjectID': sid,
        'Cluster': row['Cluster'],
        'NumWeakSkills': row['NumWeakSkills'],
        'ProblemID': row['ProblemID'],
        'Score': row['Score'],
        'ScorePct': row['ScorePct'],
        'IsPerfect': row['IsPerfect'],
        'Baseline_KCTags': row['Baseline_KCTags'],
        'Enriched_KCTags': row['Enriched_KCTags'],
        'Baseline_GAP_Count': row['Baseline_GAP_Count'],
        'Enriched_GAP_Count': row['Enriched_GAP_Count'],

        # Old metrics (kept for comparison)
        'Old_Baseline_WeakOverlap_F1': round(old_bl_f1, 4),
        'Old_Enriched_WeakOverlap_F1': round(old_en_f1, 4),

        # NEW corrected metrics
        'Corrected_Baseline_WeakOverlap_F1': round(new_bl_f1, 4),
        'Corrected_Enriched_WeakOverlap_F1': round(new_en_f1, 4),
        'Corrected_Baseline_Precision': round(bl_p, 4),
        'Corrected_Baseline_Recall': round(bl_r, 4),
        'Corrected_Enriched_Precision': round(en_p, 4),
        'Corrected_Enriched_Recall': round(en_r, 4),

        # Context for verification
        'Num_Testable_Weak': len(testable_weak),
        'Testable_Weak_Skills': str(sorted(testable_weak)),
        'Problem_Required_Skills': str(sorted(required)),

        # Relevance (unchanged from original batch run)
        'Baseline_Relevance_F1': row['Baseline_Relevance_F1'],
        'Enriched_Relevance_F1': row['Enriched_Relevance_F1'],

        # Timing
        'Baseline_TimeSec': row['Baseline_TimeSec'],
        'Enriched_TimeSec': row['Enriched_TimeSec'],
    })

corrected_df = pd.DataFrame(results)
print(f"Recomputed {len(corrected_df)} rows")
print(f"Rows with testable weak skills > 0: {(corrected_df['Num_Testable_Weak'] > 0).sum()}")
print(f"Rows where old and corrected differ: {(corrected_df['Old_Baseline_WeakOverlap_F1'] != corrected_df['Corrected_Baseline_WeakOverlap_F1']).sum()}")

In [ ]:
# Use rows with any weak skills for old metric baseline
has_weak = corrected_df[corrected_df['NumWeakSkills'] > 0]
# Use rows with testable weak skills for corrected metric
has_testable = corrected_df[corrected_df['Num_Testable_Weak'] > 0]

print("=" * 70)
print("OVERALL COMPARISON: OLD vs CORRECTED METRIC")
print("=" * 70)

print(f"\nRows with any weak skills: {len(has_weak)}")
print(f"Rows with testable weak skills: {len(has_testable)}")
print(f"Rows where problem doesn't test any weak skill: {len(has_weak) - len(has_testable)}")

print(f"\n{'Metric':<40} {'Baseline':<12} {'Enriched':<12} {'Delta':<12}")
print("-" * 70)

old_bl_mean = has_weak['Old_Baseline_WeakOverlap_F1'].mean()
old_en_mean = has_weak['Old_Enriched_WeakOverlap_F1'].mean()
print(f"{'Old WeakOverlap F1 (all weak skills)':<40} {old_bl_mean:<12.4f} {old_en_mean:<12.4f} {old_en_mean-old_bl_mean:<+12.4f}")

new_bl_mean = has_testable['Corrected_Baseline_WeakOverlap_F1'].mean()
new_en_mean = has_testable['Corrected_Enriched_WeakOverlap_F1'].mean()
print(f"{'Corrected F1 (testable weak only)':<40} {new_bl_mean:<12.4f} {new_en_mean:<12.4f} {new_en_mean-new_bl_mean:<+12.4f}")

rel_old = ((old_en_mean - old_bl_mean) / old_bl_mean * 100) if old_bl_mean > 0 else 0
rel_new = ((new_en_mean - new_bl_mean) / new_bl_mean * 100) if new_bl_mean > 0 else 0
print(f"\n{'Old relative improvement:':<40} {rel_old:+.1f}%")
print(f"{'Corrected relative improvement:':<40} {rel_new:+.1f}%")

In [ ]:
print("=" * 70)
print("PER-CLUSTER BREAKDOWN")
print("=" * 70)

for cluster in ['Struggling', 'Average', 'High Performer']:
    c_all = corrected_df[corrected_df['Cluster'] == cluster]
    c_weak = c_all[c_all['NumWeakSkills'] > 0]
    c_testable = c_all[c_all['Num_Testable_Weak'] > 0]

    print(f"\n--- {cluster} ({c_all['SubjectID'].nunique()} students, {len(c_all)} problems) ---")

    if len(c_testable) == 0:
        print("  No testable weak skills — summary skipped")
        continue

    old_bl = c_weak['Old_Baseline_WeakOverlap_F1'].mean()
    old_en = c_weak['Old_Enriched_WeakOverlap_F1'].mean()
    new_bl = c_testable['Corrected_Baseline_WeakOverlap_F1'].mean()
    new_en = c_testable['Corrected_Enriched_WeakOverlap_F1'].mean()

    print(f"  Rows with testable weak skills: {len(c_testable)}")
    print(f"  Old F1:       Baseline={old_bl:.4f}  Enriched={old_en:.4f}  Delta={old_en-old_bl:+.4f}")
    print(f"  Corrected F1: Baseline={new_bl:.4f}  Enriched={new_en:.4f}  Delta={new_en-new_bl:+.4f}")

In [ ]:
student_summary = []

for sid in corrected_df['SubjectID'].unique():
    s = corrected_df[corrected_df['SubjectID'] == sid]
    s_testable = s[s['Num_Testable_Weak'] > 0]

    row = {
        'SubjectID': sid,
        'Cluster': s['Cluster'].iloc[0],
        'NumWeakSkills': s['NumWeakSkills'].iloc[0],
        'TotalProblems': len(s),
        'TestableProblems': len(s_testable),
        'Old_Baseline_F1': round(s['Old_Baseline_WeakOverlap_F1'].mean(), 4),
        'Old_Enriched_F1': round(s['Old_Enriched_WeakOverlap_F1'].mean(), 4),
        'Old_Delta': round(s['Old_Enriched_WeakOverlap_F1'].mean() - s['Old_Baseline_WeakOverlap_F1'].mean(), 4),
    }

    if len(s_testable) > 0:
        row['Corrected_Baseline_F1'] = round(s_testable['Corrected_Baseline_WeakOverlap_F1'].mean(), 4)
        row['Corrected_Enriched_F1'] = round(s_testable['Corrected_Enriched_WeakOverlap_F1'].mean(), 4)
        row['Corrected_Delta'] = round(row['Corrected_Enriched_F1'] - row['Corrected_Baseline_F1'], 4)
    else:
        row['Corrected_Baseline_F1'] = 0.0
        row['Corrected_Enriched_F1'] = 0.0
        row['Corrected_Delta'] = 0.0

    student_summary.append(row)

summary_df = pd.DataFrame(student_summary).sort_values(['Cluster', 'Corrected_Delta'], ascending=[True, False])

print("=" * 70)
print("PER-STUDENT SUMMARY: OLD vs CORRECTED")
print("=" * 70)

for cluster in ['Struggling', 'Average', 'High Performer']:
    print(f"\n--- {cluster} ---")
    c = summary_df[summary_df['Cluster'] == cluster]
    for _, r in c.iterrows():
        weak = int(r['NumWeakSkills'])
        if weak == 0:
            print(f"  {int(r['SubjectID'])}: 0 weak skills — no change")
            continue
        print(f"  {int(r['SubjectID'])}: {weak} weak, "
              f"Old delta={r['Old_Delta']:+.4f}, "
              f"Corrected delta={r['Corrected_Delta']:+.4f}, "
              f"Testable problems={int(r['TestableProblems'])}/{int(r['TotalProblems'])}")

# Win/loss count on corrected metric
with_weak = summary_df[summary_df['NumWeakSkills'] > 0]
improved = (with_weak['Corrected_Delta'] > 0).sum()
same = (with_weak['Corrected_Delta'] == 0).sum()
worse = (with_weak['Corrected_Delta'] < 0).sum()

print(f"\nOverall students with weak skills: {len(with_weak)}")
print(f"Improved: {improved} | Same: {same} | Worse: {worse}")
if len(with_weak) > 0:
    print(f"Win rate: {improved}/{len(with_weak)} = {improved/len(with_weak)*100:.0f}%")

In [ ]:
results_dir = ROOT / 'results' / '06_batch_30students'

# Save corrected full results
corrected_df.to_csv(results_dir / 'batch_comparison_30students_corrected.csv', index=False)

# Save corrected per-student summary
summary_df.to_csv(results_dir / 'per_student_summary_corrected.csv', index=False)

print(f"Saved corrected results to {results_dir}")
print(f"  - batch_comparison_30students_corrected.csv ({len(corrected_df)} rows)")
print(f"  - per_student_summary_corrected.csv ({len(summary_df)} rows)")